In [2]:
import json
import os
import sys

import joblib
import numpy as np
import pandas as pd

sys.path.append("..")
from src.stats import rate_table

APP_DATA = "../app_data"
os.makedirs(APP_DATA, exist_ok=True)

df = pd.read_csv("../data/processed/loans_clean.csv")
df["Segment"] = np.load("../data/processed/labels_kmeans.npy")
df["Anomaly"] = np.load("../data/processed/anomaly_flags.npy")

try:
    with open("../models/segment_names.json") as file:
        names = {int(key): value for key, value in json.load(file).items()}
except FileNotFoundError:
    names = {}
for segment in sorted(df["Segment"].unique()):
    names.setdefault(int(segment), f"Segment {segment}")
print(names)

{0: 'Established earners', 1: 'Stretched (high debt-to-income)', 2: 'Prime credit', 3: 'Smaller loans, lower income', 4: 'Higher income, short tenure'}


In [3]:
numeric = ["Income", "LoanAmount", "DTI", "CreditScore", "EmploymentLength"]

profile = df.groupby("Segment")[numeric].median().round(1)
profile.insert(0, "name", pd.Series(names))
profile.insert(1, "share", df["Segment"].value_counts(normalize=True).sort_index().round(4))
rates = rate_table(df, "Segment").set_index("Segment")
profile = profile.join(rates[["loans", "defaults", "default_rate", "ci_low", "ci_high"]])
profile.to_csv(f"{APP_DATA}/segment_profile.csv")

with open("../models/feature_info.json") as file:
    feature_names = json.load(file)["feature_names"]
X_scaled = np.load("../data/processed/X_scaled.npy", mmap_mode="r")
signature = pd.DataFrame(np.asarray(X_scaled[:, :5]), columns=feature_names[:5]).groupby(df["Segment"].to_numpy()).mean()
signature.index.name = "Segment"
signature.round(3).to_csv(f"{APP_DATA}/segment_signature.csv")
profile

,name,share,Income,LoanAmount,DTI,CreditScore,EmploymentLength,loans,defaults,default_rate,ci_low,ci_high
Segment,,,,,,,,,,,,
0,Established earners,0.2560,88000.0,17700.0,15.1,687.0,10.0,345055,66134,0.191662,0.190352,0.192979
1,Stretched (high debt-to-income),0.2127,51000.0,12000.0,28.5,687.0,8.0,286607,78669,0.274484,0.272853,0.276121
2,Prime credit,0.1156,75000.0,14000.0,14.4,757.0,7.0,155730,14869,0.095479,0.094030,0.096949
3,"Smaller loans, lower income",0.2181,41000.0,5650.0,15.2,682.0,4.0,293888,53951,0.183577,0.182181,0.184981
4,"Higher income, short tenure",0.1977,85000.0,17500.0,15.8,687.0,2.0,266401,55626,0.208806,0.207266,0.210353


In [4]:
for column, file_name in [("Purpose", "mix_purpose"), ("HomeOwnership", "mix_home"), ("State", "mix_state")]:
    pd.crosstab(df["Segment"], df[column]).to_csv(f"{APP_DATA}/{file_name}.csv")

rate_table(df, "State").to_csv(f"{APP_DATA}/state_summary.csv", index=False)
rate_table(df, ["Segment", "IssueYear"]).to_csv(f"{APP_DATA}/year_summary.csv", index=False)

In [5]:
anomaly_profile = df.groupby("Anomaly")[numeric].median().round(1)
anomaly_rates = rate_table(df, "Anomaly").set_index("Anomaly")
anomaly_profile.join(anomaly_rates).to_csv(f"{APP_DATA}/anomaly_profile.csv")

by_segment = df.groupby("Segment")["Anomaly"].agg(flagged="sum", loans="size")
by_segment["flagged_share"] = by_segment["flagged"] / by_segment["loans"]
by_segment.to_csv(f"{APP_DATA}/anomaly_by_segment.csv")

In [6]:
keep = ["Income", "DTI", "LoanAmount", "CreditScore", "EmploymentLength", "Purpose", "HomeOwnership",
        "State", "IssueYear", "Default", "Segment", "Anomaly"]
df.sample(100_000, random_state=42)[keep].to_csv(f"{APP_DATA}/loans_sample.csv", index=False)

sample_idx = np.load("../data/processed/sample_idx.npy")
umap_embedding = np.load("../data/processed/umap_embedding.npy")
umap_table = df.iloc[sample_idx][["Segment", "Anomaly", "CreditScore", "Income", "DTI", "LoanAmount", "EmploymentLength"]].copy()
umap_table.insert(0, "UMAP2", umap_embedding[:, 1])
umap_table.insert(0, "UMAP1", umap_embedding[:, 0])
umap_table.to_csv(f"{APP_DATA}/umap_sample.csv", index=False)

In [7]:
pca = joblib.load("../models/pca.joblib")
variance = pd.DataFrame({"component": np.arange(1, len(pca.explained_variance_ratio_) + 1),
                         "explained": pca.explained_variance_ratio_})
variance["cumulative"] = variance["explained"].cumsum()
variance.round(4).to_csv(f"{APP_DATA}/pca_variance.csv", index=False)

for file_name in ["algorithm_comparison.csv", "default_adjusted_by_segment.csv"]:
    source = f"../models/{file_name}"
    if os.path.exists(source):
        pd.read_csv(source, index_col=0).to_csv(f"{APP_DATA}/{file_name}")
    else:
        print("Missing (create it first):", source)

total = 0
for file_name in sorted(os.listdir(APP_DATA)):
    size = os.path.getsize(f"{APP_DATA}/{file_name}") / 1e6
    total += size
    print(f"{file_name:32s} {size:6.2f} MB")
print(f"Total: {total:.1f} MB")

algorithm_comparison.csv           0.00 MB
anomaly_by_segment.csv             0.00 MB
anomaly_profile.csv                0.00 MB
default_adjusted_by_segment.csv    0.00 MB
loans_sample.csv                   7.35 MB
mix_home.csv                       0.00 MB
mix_purpose.csv                    0.00 MB
mix_state.csv                      0.00 MB
pca_variance.csv                   0.00 MB
segment_profile.csv                0.00 MB
segment_signature.csv              0.00 MB
state_summary.csv                  0.00 MB
umap_sample.csv                    1.20 MB
year_summary.csv                   0.00 MB
Total: 8.6 MB
